In [ ]:
# ============================================================
# Property Sales - Raw Load and Observation Merge
# ============================================================
#
# Source grain:
#     One row from one RealEstate.com.au scrape observation.
#
# Target grain:
#     One row per listing URL and scrape timestamp.
#
# Important:
#     detailUrl identifies a listing, not an observation.
#     The same listing may legitimately appear in multiple scrapes.
#
# This notebook:
#     1. Reads all matching CSV files as strings.
#     2. Validates the expected source structure.
#     3. Applies only minimal Raw-layer typing.
#     4. Generates deterministic record and observation hashes.
#     5. Removes duplicate imports of the same observation.
#     6. Merges new observations into Raw.property_sales_data.
#
# Existing legacy targets that do not contain ObservationKey are
# rebuilt automatically from the available CSV source archive.
# ============================================================

In [1]:
# ============================================================
# 1. Imports
# ============================================================

from delta.tables import DeltaTable

from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
)

StatementMeta(, de75ed7f-c49e-444d-a3b1-70a05a3fb519, 3, Finished, Available, Finished, False)

In [3]:
# ============================================================
# 2. Configuration
# ============================================================

SOURCE_PATH = "Files/rea-sold-*.csv"

TARGET_SCHEMA = "Raw"
TARGET_TABLE_NAME = "property_sales_data"
TARGET_TABLE = f"{TARGET_SCHEMA}.{TARGET_TABLE_NAME}"

# Rebuild manually when required.
#
# Normally leave this as False. The notebook automatically rebuilds
# a legacy target that does not yet contain ObservationKey.
FORCE_REBUILD = False

# When True, fail if a required source column is absent.
FAIL_ON_MISSING_REQUIRED_COLUMNS = True


# Columns expected from the scraper CSV output.
#
# All source fields are initially read as strings. Numeric fields are
# typed only after the source has been landed into a DataFrame.
EXPECTED_SOURCE_COLUMNS = [
    "sourceUrl",
    "pageNumber",
    "ordinalOnPage",
    "price",
    "priceValue",
    "address",
    "detailPath",
    "detailUrl",
    "bedrooms",
    "bathrooms",
    "carSpaces",
    "landSize",
    "landSizeSqm",
    "propertyType",
    "soldDate",
    "soldDateISO",
    "scrapedAt",
]

# A valid listing observation needs at least one usable listing URL/path.
REQUIRED_IDENTITY_COLUMNS = [
    "detailUrl",
    "detailPath",
]

StatementMeta(, de75ed7f-c49e-444d-a3b1-70a05a3fb519, 5, Finished, Available, Finished, False)

In [4]:
# ============================================================
# 3. Spark configuration
# ============================================================

# Keep Raw ingestion permissive. Malformed values should become NULL
# rather than aborting the entire load.
spark.conf.set("spark.sql.ansi.enabled", "false")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

# Permit new audit columns to be added during Delta operations.
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}")

StatementMeta(, de75ed7f-c49e-444d-a3b1-70a05a3fb519, 6, Finished, Available, Finished, False)

DataFrame[]

In [5]:
# ============================================================
# 4. Explicit CSV schema
# ============================================================

# Reading all source fields as strings avoids schema drift between files
# and prevents Spark from assigning incompatible inferred data types.
source_schema = StructType(
    [
        StructField(column_name, StringType(), True)
        for column_name in EXPECTED_SOURCE_COLUMNS
    ]
)

StatementMeta(, de75ed7f-c49e-444d-a3b1-70a05a3fb519, 7, Finished, Available, Finished, False)

In [6]:
# ============================================================
# 5. Helper expressions
# ============================================================

def normalise_string(column_name):
    """
    Apply minimal Raw-layer string normalisation.

    This deliberately does not perform business cleaning. It only:
      - casts to string
      - removes surrounding whitespace
      - converts blank strings to NULL

    More invasive repair belongs in the Cleaned notebook.
    """
    value = F.trim(F.col(column_name).cast("string"))

    return (
        F.when(value.isNull() | (value == ""), F.lit(None).cast("string"))
        .otherwise(value)
    )


def safe_integer(column_name):
    """
    Parse a source field as INTEGER without failing the load.
    """
    value = F.regexp_replace(
        normalise_string(column_name),
        ",",
        "",
    )

    return value.cast("int")


def safe_long(column_name):
    """
    Parse a source field as BIGINT without failing the load.
    """
    value = F.regexp_replace(
        normalise_string(column_name),
        ",",
        "",
    )

    return value.cast("long")


def table_exists(table_name):
    """
    Return True when the specified Spark catalog table exists.
    """
    return spark.catalog.tableExists(table_name)

StatementMeta(, de75ed7f-c49e-444d-a3b1-70a05a3fb519, 8, Finished, Available, Finished, False)

In [7]:
# ============================================================
# 6. Read source CSV files
# ============================================================

df_source = (
    spark.read
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .option("multiLine", "false")
    .option("quote", '"')
    .option("escape", '"')
    .option("enforceSchema", "false")
    .schema(source_schema)
    .csv(SOURCE_PATH)
    .withColumn("SourceFile", F.input_file_name())
)

source_file_count = (
    df_source
    .select("SourceFile")
    .where(F.col("SourceFile").isNotNull())
    .distinct()
    .count()
)

source_row_count = df_source.count()

print(f"Source path: {SOURCE_PATH}")
print(f"Source files discovered: {source_file_count:,}")
print(f"Source rows read: {source_row_count:,}")

if source_file_count == 0:
    raise RuntimeError(
        f"No source files were found for path pattern: {SOURCE_PATH}"
    )

if source_row_count == 0:
    raise RuntimeError(
        f"Source files were found, but no data rows were read from: {SOURCE_PATH}"
    )

StatementMeta(, de75ed7f-c49e-444d-a3b1-70a05a3fb519, 9, Finished, Available, Finished, False)

Source path: Files/rea-sold-*.csv
Source files discovered: 359
Source rows read: 347,903


In [8]:
# ============================================================
# 7. Validate source columns
# ============================================================

available_columns = set(df_source.columns)

missing_expected_columns = [
    column_name
    for column_name in EXPECTED_SOURCE_COLUMNS
    if column_name not in available_columns
]

missing_identity_columns = [
    column_name
    for column_name in REQUIRED_IDENTITY_COLUMNS
    if column_name not in available_columns
]

if missing_expected_columns:
    message = (
        "The following expected source columns are missing: "
        + ", ".join(missing_expected_columns)
    )

    if FAIL_ON_MISSING_REQUIRED_COLUMNS:
        raise RuntimeError(message)

    print(f"WARNING: {message}")

if len(missing_identity_columns) == len(REQUIRED_IDENTITY_COLUMNS):
    raise RuntimeError(
        "Neither detailUrl nor detailPath is available. "
        "A listing observation identity cannot be generated."
    )

StatementMeta(, de75ed7f-c49e-444d-a3b1-70a05a3fb519, 10, Finished, Available, Finished, False)

In [9]:
# ============================================================
# 8. Apply minimal Raw-layer typing
# ============================================================

df_typed = (
    df_source

    # Source and listing text
    .withColumn("sourceUrl", normalise_string("sourceUrl"))
    .withColumn("price", normalise_string("price"))
    .withColumn("address", normalise_string("address"))
    .withColumn("detailPath", normalise_string("detailPath"))
    .withColumn("detailUrl", normalise_string("detailUrl"))
    .withColumn("landSize", normalise_string("landSize"))
    .withColumn("propertyType", normalise_string("propertyType"))
    .withColumn("soldDate", normalise_string("soldDate"))
    .withColumn("scrapedAt", normalise_string("scrapedAt"))

    # Source ordering
    .withColumn("pageNumber", safe_integer("pageNumber"))
    .withColumn("ordinalOnPage", safe_integer("ordinalOnPage"))

    # Numeric source attributes
    .withColumn("priceValue", safe_long("priceValue"))
    .withColumn("bedrooms", safe_integer("bedrooms"))
    .withColumn("bathrooms", safe_integer("bathrooms"))
    .withColumn("carSpaces", safe_integer("carSpaces"))
    .withColumn("landSizeSqm", safe_integer("landSizeSqm"))

    # Parse the explicitly supplied ISO/date field while retaining
    # soldDate as the original source text.
    .withColumn(
        "soldDateISO",
        F.coalesce(
            F.to_date(normalise_string("soldDateISO"), "d/MM/yyyy"),
            F.to_date(normalise_string("soldDateISO"), "dd/MM/yyyy"),
            F.to_date(normalise_string("soldDateISO"), "yyyy-MM-dd"),
        ),
    )

    # File-level provenance
    .withColumn("SourceFile", normalise_string("SourceFile"))
    .withColumn("LoadedAt", F.current_timestamp())
    .withColumn("LoadDate", F.current_date())
)

StatementMeta(, de75ed7f-c49e-444d-a3b1-70a05a3fb519, 11, Finished, Available, Finished, False)

In [10]:
# ============================================================
# 9. Construct listing identity
# ============================================================

# Prefer the canonical absolute URL.
#
# When only detailPath exists, retain it as the identity rather than
# dropping the row. The Cleaned notebook can reconstruct the full URL.
df_typed = (
    df_typed
    .withColumn(
        "_ListingIdentity",
        F.coalesce(
            F.lower(F.trim(F.col("detailUrl"))),
            F.lower(F.trim(F.col("detailPath"))),
        ),
    )
    .filter(F.col("_ListingIdentity").isNotNull())
)

StatementMeta(, de75ed7f-c49e-444d-a3b1-70a05a3fb519, 12, Finished, Available, Finished, False)

In [11]:
# ============================================================
# 10. Generate deterministic hashes
# ============================================================

# RecordHash describes the complete source content of the observation.
#
# SourceFile, LoadedAt and LoadDate are deliberately excluded. Importing
# the same CSV under a different filename must not create another record.
hash_source_columns = [
    "sourceUrl",
    "pageNumber",
    "ordinalOnPage",
    "price",
    "priceValue",
    "address",
    "detailPath",
    "detailUrl",
    "bedrooms",
    "bathrooms",
    "carSpaces",
    "landSize",
    "landSizeSqm",
    "propertyType",
    "soldDate",
    "soldDateISO",
    "scrapedAt",
]

record_hash_expression = F.sha2(
    F.to_json(
        F.struct(
            *[
                F.col(column_name).alias(column_name)
                for column_name in hash_source_columns
            ]
        ),
        options={"ignoreNullFields": "false"},
    ),
    256,
)

df_hashed = df_typed.withColumn("RecordHash", record_hash_expression)

# ObservationKey identifies a particular listing at a particular scrape.
#
# Normal case:
#     listing URL/path + scrapedAt
#
# Fallback:
#     listing URL/path + RecordHash
#
# The fallback preserves distinct source observations when scrapedAt is
# absent, while still making repeated imports idempotent.
df_hashed = df_hashed.withColumn(
    "ObservationKey",
    F.sha2(
        F.concat_ws(
            "||",
            F.col("_ListingIdentity"),
            F.coalesce(
                F.col("scrapedAt"),
                F.concat(F.lit("NO_SCRAPE_TIMESTAMP:"), F.col("RecordHash")),
            ),
        ),
        256,
    ),
)

StatementMeta(, de75ed7f-c49e-444d-a3b1-70a05a3fb519, 13, Finished, Available, Finished, False)

In [12]:
# ============================================================
# 11. Remove repeated imports of identical observations
# ============================================================

# Multiple copies of the same exported CSV, or overlapping source glob
# results, may contain the same observation. Keep one deterministic copy.
df_incoming = (
    df_hashed
    .dropDuplicates(["ObservationKey"])
    .drop("_ListingIdentity")
)

incoming_row_count = df_incoming.count()
duplicate_source_row_count = source_row_count - incoming_row_count

print(f"Unique incoming observations: {incoming_row_count:,}")
print(
    "Repeated source observations removed: "
    f"{duplicate_source_row_count:,}"
)

StatementMeta(, de75ed7f-c49e-444d-a3b1-70a05a3fb519, 14, Finished, Available, Finished, False)

Unique incoming observations: 347,887
Repeated source observations removed: 16


In [13]:
# ============================================================
# 12. Determine target migration mode
# ============================================================

target_exists = table_exists(TARGET_TABLE)
legacy_target_detected = False

if target_exists:
    target_columns = set(spark.table(TARGET_TABLE).columns)

    legacy_target_detected = (
        "ObservationKey" not in target_columns
        or "RecordHash" not in target_columns
    )

should_rebuild = (
    FORCE_REBUILD
    or not target_exists
    or legacy_target_detected
)

if FORCE_REBUILD:
    print("Target rebuild requested by FORCE_REBUILD=True.")
elif not target_exists:
    print(f"Target table does not exist and will be created: {TARGET_TABLE}")
elif legacy_target_detected:
    print(
        f"Legacy target detected: {TARGET_TABLE}. "
        "The table does not contain the observation-level keys and "
        "will be rebuilt from the source CSV archive."
    )
else:
    print(
        f"Observation-level target detected: {TARGET_TABLE}. "
        "New observations will be merged."
    )

StatementMeta(, de75ed7f-c49e-444d-a3b1-70a05a3fb519, 15, Finished, Available, Finished, False)

Legacy target detected: Raw.property_sales_data. The table does not contain the observation-level keys and will be rebuilt from the source CSV archive.


In [14]:
# ============================================================
# 13. Create, rebuild, or merge the target
# ============================================================

if should_rebuild:
    (
        df_incoming.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TARGET_TABLE)
    )

    load_action = "REBUILT"

else:
    target = DeltaTable.forName(spark, TARGET_TABLE)

    (
        target.alias("t")
        .merge(
            df_incoming.alias("s"),
            "t.ObservationKey = s.ObservationKey",
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    load_action = "MERGED"


print(f"{load_action}: {TARGET_TABLE}")

StatementMeta(, de75ed7f-c49e-444d-a3b1-70a05a3fb519, 16, Finished, Available, Finished, False)

REBUILT: Raw.property_sales_data


In [15]:
# ============================================================
# 14. Capture Delta operation metrics
# ============================================================

try:
    operation_history = (
        DeltaTable.forName(spark, TARGET_TABLE)
        .history(1)
        .select(
            "timestamp",
            "operation",
            "operationParameters",
            "operationMetrics",
        )
    )

    operation_history.show(truncate=False)

except Exception as history_error:
    print(
        "WARNING: Delta operation history could not be displayed: "
        f"{history_error}"
    )

StatementMeta(, de75ed7f-c49e-444d-a3b1-70a05a3fb519, 17, Finished, Available, Finished, False)

+-----------------------+---------------------------------+----------------------------------------------------------------------------------------------+--------------------------------------------------------------------+
|timestamp              |operation                        |operationParameters                                                                           |operationMetrics                                                    |
+-----------------------+---------------------------------+----------------------------------------------------------------------------------------------+--------------------------------------------------------------------+
|2026-07-11 15:38:18.525|CREATE OR REPLACE TABLE AS SELECT|{partitionBy -> [], clusterBy -> [], description -> NULL, isManaged -> true, properties -> {}}|{numFiles -> 1, numOutputRows -> 347887, numOutputBytes -> 71312847}|
+-----------------------+---------------------------------+---------------------------------------------

In [16]:
# ============================================================
# 15. Validate the resulting Raw table
# ============================================================

target_summary = spark.sql(
    f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT ObservationKey) AS distinct_observation_count,
        COUNT(DISTINCT detailUrl) AS distinct_detail_url_count,
        COUNT(DISTINCT detailPath) AS distinct_detail_path_count,

        SUM(
            CASE
                WHEN ObservationKey IS NULL THEN 1
                ELSE 0
            END
        ) AS null_observation_key_count,

        SUM(
            CASE
                WHEN detailUrl IS NULL AND detailPath IS NULL THEN 1
                ELSE 0
            END
        ) AS missing_listing_identity_count,

        SUM(
            CASE
                WHEN scrapedAt IS NULL THEN 1
                ELSE 0
            END
        ) AS missing_scraped_at_count,

        MIN(scrapedAt) AS minimum_scraped_at,
        MAX(scrapedAt) AS maximum_scraped_at,
        MIN(LoadedAt) AS minimum_loaded_at,
        MAX(LoadedAt) AS maximum_loaded_at

    FROM {TARGET_TABLE}
    """
)

target_summary.show(truncate=False)

StatementMeta(, de75ed7f-c49e-444d-a3b1-70a05a3fb519, 18, Finished, Available, Finished, False)

+---------+--------------------------+-------------------------+--------------------------+--------------------------+------------------------------+------------------------+------------------------+------------------------+--------------------------+--------------------------+
|row_count|distinct_observation_count|distinct_detail_url_count|distinct_detail_path_count|null_observation_key_count|missing_listing_identity_count|missing_scraped_at_count|minimum_scraped_at      |maximum_scraped_at      |minimum_loaded_at         |maximum_loaded_at         |
+---------+--------------------------+-------------------------+--------------------------+--------------------------+------------------------------+------------------------+------------------------+------------------------+--------------------------+--------------------------+
|347887   |347887                    |296422                   |296422                    |0                         |0                             |0             

In [17]:
# ============================================================
# 16. Assert target invariants
# ============================================================

validation = target_summary.first()

if validation["null_observation_key_count"] != 0:
    raise RuntimeError(
        "Raw target validation failed: ObservationKey contains NULL values."
    )

if validation["missing_listing_identity_count"] != 0:
    raise RuntimeError(
        "Raw target validation failed: rows exist without detailUrl "
        "or detailPath."
    )

if (
    validation["row_count"]
    != validation["distinct_observation_count"]
):
    raise RuntimeError(
        "Raw target validation failed: ObservationKey is not unique."
    )

StatementMeta(, de75ed7f-c49e-444d-a3b1-70a05a3fb519, 19, Finished, Available, Finished, False)

In [18]:
# ============================================================
# 17. Profile repeated listing observations
# ============================================================

print("Listings with the largest number of retained scrape observations:")

spark.sql(
    f"""
    SELECT
        COALESCE(detailUrl, detailPath) AS ListingReference,
        COUNT(*) AS ObservationCount,
        MIN(scrapedAt) AS FirstScrapedAt,
        MAX(scrapedAt) AS LastScrapedAt,
        COUNT(DISTINCT RecordHash) AS DistinctRecordVersions

    FROM {TARGET_TABLE}

    GROUP BY
        COALESCE(detailUrl, detailPath)

    HAVING COUNT(*) > 1

    ORDER BY
        ObservationCount DESC,
        ListingReference

    LIMIT 50
    """
).show(truncate=False)

StatementMeta(, de75ed7f-c49e-444d-a3b1-70a05a3fb519, 20, Finished, Available, Finished, False)

Listings with the largest number of retained scrape observations:
+---------------------------------------------------------------------+----------------+------------------------+------------------------+----------------------+
|ListingReference                                                     |ObservationCount|FirstScrapedAt          |LastScrapedAt           |DistinctRecordVersions|
+---------------------------------------------------------------------+----------------+------------------------+------------------------+----------------------+
|https://www.realestate.com.au/sold/property-house-wa-medina-150007780|11              |2026-07-02T17:10:53.136Z|2026-07-02T19:00:23.109Z|11                    |
|https://www.realestate.com.au/sold/property-house-wa-medina-150065024|11              |2026-07-02T17:10:48.705Z|2026-07-02T19:00:23.109Z|11                    |
|https://www.realestate.com.au/sold/property-house-wa-medina-150118112|11              |2026-07-02T17:10:48.705Z|2026-07-02T

In [19]:
# ============================================================
# 18. Final status
# ============================================================

print(
    f"""
Raw property-sales load complete.

Target table:              {TARGET_TABLE}
Load action:               {load_action}
Source files:              {source_file_count:,}
Source rows:               {source_row_count:,}
Incoming observations:     {incoming_row_count:,}
Source duplicates removed: {duplicate_source_row_count:,}
Target rows:               {validation["row_count"]:,}
Distinct listing URLs:     {validation["distinct_detail_url_count"]:,}
"""
)

StatementMeta(, de75ed7f-c49e-444d-a3b1-70a05a3fb519, 21, Finished, Available, Finished, False)


Raw property-sales load complete.

Target table:              Raw.property_sales_data
Load action:               REBUILT
Source files:              359
Source rows:               347,903
Incoming observations:     347,887
Source duplicates removed: 16
Target rows:               347,887
Distinct listing URLs:     296,422

